# ACTIVIDAD SESIÓN 4 (RESUELTA): PROCESAMIENTO DE DATOS CON SPARK SQL Y DATAFRAMES

Este cuaderno resuelve paso a paso la actividad usando **PySpark**.  
Dataset objetivo: `carreras.json` (con **fallback** a datos embebidos si el archivo no está disponible).

## 1. Creación de la sesión Spark

In [1]:
from pyspark.sql import SparkSession

# Crear sesión de Spark

spark = SparkSession.builder.appName("AnalisisCarreras").getOrCreate()
spark
print("Spark inicializado")
print("Versión:", spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/09/05 21:37:12 WARN Utils: Your hostname, Matheus-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.94 instead (on interface en0)
25/09/05 21:37:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/05 21:37:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark inicializado
Versión: 4.0.0


## 2. Carga de datos en un DataFrame y vista previa

In [2]:
from pyspark.sql import functions as F, types as T

# Intentar leer desde carreras.json; si no existe, usar datos embebidos
try:
    df = spark.read.json("carreras.json", multiLine=True)
    origen = "archivo 'carreras.json'"
except Exception as e:
    data = [
        {"id": 1, "carrera": "Ingeniería Comercial", "universidad": "U. de Chile", "inscritos": 3200, "area": "Económicas y Administrativas"},
        {"id": 2, "carrera": "Derecho", "universidad": "PUC", "inscritos": 2900, "area": "Ciencias Sociales"},
        {"id": 3, "carrera": "Medicina", "universidad": "U. de Concepción", "inscritos": 2500, "area": "Salud"},
        {"id": 4, "carrera": "Psicología", "universidad": "U. de Santiago", "inscritos": 1800, "area": "Ciencias Sociales"},
        {"id": 5, "carrera": "Ingeniería Civil", "universidad": "UTFSM", "inscritos": 3100, "area": "Ingeniería y Tecnología"},
    ]
    schema = T.StructType([
        T.StructField("id", T.IntegerType()),
        T.StructField("carrera", T.StringType()),
        T.StructField("universidad", T.StringType()),
        T.StructField("inscritos", T.IntegerType()),
        T.StructField("area", T.StringType()),
    ])
    df = spark.createDataFrame(data, schema=schema)
    origen = "datos embebidos (fallback)"

print(f"Origen de datos: {origen}")
df.show(truncate=False)

25/09/05 21:40:41 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: carreras.json.
java.io.FileNotFoundException: File carreras.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfu

Origen de datos: datos embebidos (fallback)


+---+--------------------+----------------+---------+----------------------------+
|id |carrera             |universidad     |inscritos|area                        |
+---+--------------------+----------------+---------+----------------------------+
|1  |Ingeniería Comercial|U. de Chile     |3200     |Económicas y Administrativas|
|2  |Derecho             |PUC             |2900     |Ciencias Sociales           |
|3  |Medicina            |U. de Concepción|2500     |Salud                       |
|4  |Psicología          |U. de Santiago  |1800     |Ciencias Sociales           |
|5  |Ingeniería Civil    |UTFSM           |3100     |Ingeniería y Tecnología     |
+---+--------------------+----------------+---------+----------------------------+



## 3. Exploración del DataFrame (schema)

In [3]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- carrera: string (nullable = true)
 |-- universidad: string (nullable = true)
 |-- inscritos: integer (nullable = true)
 |-- area: string (nullable = true)



## 4. Consultas con Spark SQL

In [5]:
# Registrar vista temporal
df.createOrReplaceTempView("carreras")

# a) Carreras con más de 2500 inscritos
q_a = spark.sql("""
SELECT id, carrera, universidad, inscritos, area
FROM carreras
WHERE inscritos > 2500
ORDER BY inscritos DESC
""")
print("a) Carreras con > 2500 inscritos")
q_a.show(truncate=False)


a) Carreras con > 2500 inscritos
+---+--------------------+-----------+---------+----------------------------+
|id |carrera             |universidad|inscritos|area                        |
+---+--------------------+-----------+---------+----------------------------+
|1  |Ingeniería Comercial|U. de Chile|3200     |Económicas y Administrativas|
|5  |Ingeniería Civil    |UTFSM      |3100     |Ingeniería y Tecnología     |
|2  |Derecho             |PUC        |2900     |Ciencias Sociales           |
+---+--------------------+-----------+---------+----------------------------+



In [ ]:

# b) Cantidad de carreras por área
q_b = spark.sql("""
SELECT area, COUNT(*) AS cantidad_carreras
FROM carreras
GROUP BY area
ORDER BY cantidad_carreras DESC, area
""")
print("b) Conteo por área")
q_b.show(truncate=False)

# c) Universidades que ofrecen más de una carrera en la lista
q_c = spark.sql("""
SELECT universidad, COUNT(*) AS n_carreras
FROM carreras
GROUP BY universidad
HAVING COUNT(*) > 1
ORDER BY n_carreras DESC, universidad
""")
print("c) Universidades con más de una carrera")
q_c.show(truncate=False)

## 5. UDF de clasificación por demanda

In [ ]:
from pyspark.sql.functions import udf

@udf("string")
def clasificar_demanda(inscritos):
    if inscritos is None:
        return None
    if inscritos > 3000:
        return "Alta demanda"
    elif 2000 <= inscritos <= 3000:
        return "Media demanda"
    else:
        return "Baja demanda"

df_demanda = df.withColumn("demanda", clasificar_demanda(F.col("inscritos")))
df_demanda.show(truncate=False)

## 6. Guardado de datos en Parquet

In [ ]:
# Guardar como Parquet (sobrescribir si existe)
output_path = "carreras_procesadas.parquet"
df_demanda.write.mode("overwrite").parquet(output_path)
print(f"Datos guardados en: {output_path}")

## 7. (Opcional) Detener la sesión Spark

In [ ]:
spark.stop()